# Сравнение: ground truth, MuscleMAP и Kinesis

Эта тетрадка работает из корня **`musclemap-bench`**.

Показывает:
1. Список движений, для которых есть артефакты Kinesis (`data/kinesis_artifacts/_manifest.json`).
2. **Анимацию** (MP4/GIF) одного и того же SMPL-X движения: GT · MuscleMAP · Kinesis, кости по активациям.
3. Таблицу метрик L1 (MAE, RMSE, DTW-MAE, Pearson r, R²) для MuscleMAP и Kinesis относительно ground truth на **42 сопоставимых мышцах ног**.

Предварительно:
```bash
conda activate musclemap-bench
python precompute/run_kinesis.py --config config.yaml --test-split-only  # если ещё не запускали
```

In [21]:
from __future__ import annotations

import json
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Video, display, Markdown
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config.yaml").exists() and (REPO_ROOT.parent / "config.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.align import align_lengths, build_mapping_kinesis_artifacts
from src.body_visualization import (
    create_triple_activation_animation,
    save_animation_media,
)
from src.inference import run_kinesis_from_artifact, run_musclemap
from src.loaders import (
    get_benchmark_sample,
    load_config,
    load_kinesis_manifest,
    load_kinesis_muscle_names,
    load_musclemap,
    load_test_dataset,
)
from src.metrics_l1 import aggregate_l1_metrics, compute_l1_metrics
import importlib
import src.paired_eval as _paired_eval
importlib.reload(_paired_eval)
from src.paired_eval import (
    ensure_vis_examples,
    list_paired_indices,
    paired_coverage_stats,
)
from precompute.run_kinesis import RAJAGOPAL_TO_KINESIS_ACTUATOR

cfg = load_config(REPO_ROOT / "config.yaml")
DEVICE = str(cfg.get("inference", {}).get("device", "cpu"))
VIS_SEQUENCE_ID = "Arm_Swing_during_walking_clip1"  # анимация в §4
MAX_VIS = 1          # сколько примеров визуализировать
VIS_FPS = 12         # кадров/с в MP4/GIF
VIS_FRAME_STEP = 1   # 2 = каждый второй кадр (быстрее рендер)
MAX_EVAL = None      # None = все paired окна; или int для быстрого прогона
CHECKPOINT = None    # None = config paths.musclemap_checkpoint

print(f"Repo: {REPO_ROOT}")
print(f"Device: {DEVICE}")

Repo: /Users/maksimpecin/Library/CloudStorage/OneDrive-Личная/Thesis/code/musclemap-bench
Device: mps


## 1. Список движений, обработанных Kinesis

In [22]:
manifest_raw = json.loads(
    (REPO_ROOT / "data/kinesis_artifacts/_manifest.json").read_text(encoding="utf-8")
)
k_manifest = load_kinesis_manifest(cfg)

rows = []
for seq_id, info in sorted(manifest_raw.items()):
    rows.append({
        "sequence_id": seq_id,
        "status": info.get("status", "?"),
        "timing_s": info.get("timing_s"),
        "file": info.get("file"),
        "error": info.get("error"),
    })
manifest_df = pd.DataFrame(rows)
ok_df = manifest_df[manifest_df["status"] == "ok"].copy()
err_df = manifest_df[manifest_df["status"] != "ok"].copy()

display(Markdown(
    f"**Всего записей в manifest:** {len(manifest_df)} · "
    f"**успешно (ok):** {len(ok_df)} · **ошибки:** {len(err_df)}"
))
display(ok_df[["sequence_id", "timing_s", "file"]].head(20))
if len(ok_df) > 20:
    display(Markdown(f"… и ещё {len(ok_df) - 20} успешных записей (см. `ok_df`)"))
if len(err_df):
    display(Markdown("#### Ошибки precompute"))
    display(err_df[["sequence_id", "error"]].head(10))

**Всего записей в manifest:** 6141 · **успешно (ok):** 6140 · **ошибки:** 1

,sequence_id,timing_s,file
0,Act_cute_and_sitting_at_the_same_time_clip1,0.587177,Act_cute_and_sitting_at_the_same_time_clip1.npy
1,Act_cute_and_standing_at_the_same_time_clip1,0.165922,Act_cute_and_standing_at_the_same_time_clip1.npy
2,Act_cute_during_sitting_1_clip1,0.134704,Act_cute_during_sitting_1_clip1.npy
3,Act_cute_during_sitting_2_clip1,0.160632,Act_cute_during_sitting_2_clip1.npy
4,Act_cute_during_sitting_clip1,0.121738,Act_cute_during_sitting_clip1.npy
5,Act_cute_during_walking_1_clip1,0.128208,Act_cute_during_walking_1_clip1.npy
6,Act_cute_during_walking_clip1,0.140125,Act_cute_during_walking_clip1.npy
7,Act_cute_while_sitting_1_clip1,0.151516,Act_cute_while_sitting_1_clip1.npy
8,Act_cute_while_sitting_2_clip1,0.157930,Act_cute_while_sitting_2_clip1.npy
9,Act_cute_while_sitting_clip1,0.125318,Act_cute_while_sitting_clip1.npy


… и ещё 6120 успешных записей (см. `ok_df`)

#### Ошибки precompute

,sequence_id,error
6019,standing_and_Basking_in_the_sun_at_the_same_ti...,"could not broadcast input array from shape (3,..."


## 2. Paired test-окна (есть и GT, и Kinesis)

In [23]:
dataset, rajagopal_names = load_test_dataset(cfg)
k_names = load_kinesis_muscle_names(cfg)
mapping = build_mapping_kinesis_artifacts(rajagopal_names, k_names, RAJAGOPAL_TO_KINESIS_ACTUATOR)

paired_indices = list_paired_indices(dataset, k_manifest)
if MAX_EVAL is not None:
    paired_indices = paired_indices[: int(MAX_EVAL)]

coverage = paired_coverage_stats(dataset, k_manifest)
display(Markdown(
    f"- Test окон: **{coverage['n_test_windows']}**\n"
    f"- Paired окон (Kinesis): **{coverage['n_paired_windows']}**\n"
    f"- Уникальных test-последовательностей: **{coverage['n_test_sequences']}**\n"
    f"- С Kinesis: **{coverage['n_paired_sequences']}**\n"
    f"- Сопоставимых мышц: **{len(mapping.shared_names)}**"
))

model = load_musclemap(cfg, device=DEVICE, checkpoint_path=CHECKPOINT)
print(f"Loaded MuscleMAP on {DEVICE}")

INFO Loaded test dataset: 617 items from /Users/maksimpecin/Downloads/code/motionx_activations_extended


- Test окон: **617**
- Paired окон (Kinesis): **617**
- Уникальных test-последовательностей: **567**
- С Kinesis: **567**
- Сопоставимых мышц: **42**

INFO HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/config.json "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/tokenizer_config.json "HTTP/1.1 200 OK"
INFO HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json "HTTP/1.1 

Loaded MuscleMAP on mps


## 3. Метрики по каждому paired-движению

In [24]:
layer1 = cfg["layer1"]
metric_kw = dict(
    dtw_enabled=layer1["dtw_enabled"],
    dtw_radius=layer1["dtw_radius"],
    activation_threshold=layer1["activation_threshold"],
    min_active_frames=layer1["min_active_frames"],
    compute_coactivation=False,
)

def _expand_to_rajagopal(mapped: np.ndarray, raj_indices: list[int], n: int = 80) -> np.ndarray:
    out = np.zeros((mapped.shape[0], n), dtype=np.float32)
    for j, r_idx in enumerate(raj_indices):
        out[:, int(r_idx)] = mapped[:, j]
    return out

def _load_smplx_window(ds, idx: int) -> np.ndarray:
    seq_dir, start, true_T, _text = ds._items[idx]
    motion = np.load(seq_dir / "smplx_322.npy").astype(np.float32)
    return motion[int(start) : int(start) + int(true_T)]

mm_metrics_list: list[dict] = []
kin_metrics_list: list[dict] = []
per_row: list[dict] = []
examples: list[dict] = []

for idx in paired_indices:
    sample = get_benchmark_sample(dataset, idx)
    seq_id = sample.sequence_id
    ref = sample.activations

    mm = run_musclemap(model, sample.text, seq_id, device=DEVICE, ref_T=ref.shape[0])
    kin = run_kinesis_from_artifact(k_manifest[seq_id]["path"], seq_id, sample.text)

    mm_pred, ref_a = align_lengths(mm.activations, ref)
    kin_pred = kin.activations[:, mapping.kinesis_indices]
    kin_ref = ref[:, mapping.rajagopal_indices]
    kin_pred, kin_ref = align_lengths(kin_pred, kin_ref)

    mm_p = mm_pred[:, mapping.rajagopal_indices]
    kin_p = kin_pred
    ref_p = ref_a[:, mapping.rajagopal_indices]

    mm_m = compute_l1_metrics(mm_p, ref_p, mapping.shared_names, **metric_kw)
    kin_m = compute_l1_metrics(kin_p, kin_ref, mapping.shared_names, **metric_kw)
    mm_metrics_list.append(mm_m)
    kin_metrics_list.append(kin_m)

    per_row.append({
        "sequence_id": seq_id,
        "prompt": sample.text,
        "n_frames": mm_m["n_frames"],
        "mm_mae": mm_m["mae"],
        "mm_rmse": mm_m["rmse"],
        "mm_dtw_mae": mm_m.get("dtw_mae"),
        "mm_pearson_r": mm_m["pearson_r_mean"],
        "mm_r2": mm_m["r2_mean"],
        "kin_mae": kin_m["mae"],
        "kin_rmse": kin_m["rmse"],
        "kin_dtw_mae": kin_m.get("dtw_mae"),
        "kin_pearson_r": kin_m["pearson_r_mean"],
        "kin_r2": kin_m["r2_mean"],
        "kin_timing_s": float(k_manifest[seq_id].get("timing_s", float("nan"))),
        "mm_timing_s": float(mm.timing_s),
    })

    if seq_id == VIS_SEQUENCE_ID and len(examples) < MAX_VIS:
        smplx_win = _load_smplx_window(dataset, idx)
        T = min(smplx_win.shape[0], mm_pred.shape[0], kin_pred.shape[0], ref_a.shape[0])
        examples.append({
            "sequence_id": seq_id,
            "prompt": sample.text,
            "smplx": smplx_win[:T],
            "gt": ref_a[:T],
            "mm": mm_pred[:T],
            "kin_full": _expand_to_rajagopal(kin_p[:T], mapping.rajagopal_indices, len(rajagopal_names)),
        })

metrics_df = pd.DataFrame(per_row).sort_values("mm_mae").reset_index(drop=True)
agg_mm = aggregate_l1_metrics(mm_metrics_list)
agg_kin = aggregate_l1_metrics(kin_metrics_list)

summary_df = pd.DataFrame([
    {
        "method": "MuscleMAP",
        "n_windows": agg_mm.get("n_samples"),
        "mae": agg_mm.get("mae"),
        "rmse": agg_mm.get("rmse"),
        "dtw_mae": agg_mm.get("dtw_mae"),
        "pearson_r_mean": agg_mm.get("pearson_r_mean"),
        "r2_mean": agg_mm.get("r2_mean"),
        "smoothness_pred": agg_mm.get("smoothness_pred"),
    },
    {
        "method": "Kinesis",
        "n_windows": agg_kin.get("n_samples"),
        "mae": agg_kin.get("mae"),
        "rmse": agg_kin.get("rmse"),
        "dtw_mae": agg_kin.get("dtw_mae"),
        "pearson_r_mean": agg_kin.get("pearson_r_mean"),
        "r2_mean": agg_kin.get("r2_mean"),
        "smoothness_pred": agg_kin.get("smoothness_pred"),
    },
])

display(Markdown("### Сводка по всем paired-окнам (vs OpenSim GT)"))
display(summary_df.style.format({
    "mae": "{:.5f}", "rmse": "{:.5f}", "dtw_mae": "{:.5f}",
    "pearson_r_mean": "{:.4f}", "r2_mean": "{:.4f}", "smoothness_pred": "{:.5f}",
}, na_rep="—"))

display(Markdown("### По каждому движению (первые 25 строк)"))
show_cols = [
    "sequence_id", "n_frames",
    "mm_mae", "kin_mae", "mm_rmse", "kin_rmse",
    "mm_pearson_r", "kin_pearson_r", "mm_r2", "kin_r2",
]
display(metrics_df[show_cols].head(25).style.format({
    c: "{:.5f}" for c in show_cols if c not in ("sequence_id", "n_frames")
}))

examples = ensure_vis_examples(
    examples,
    VIS_SEQUENCE_ID,
    max_vis=MAX_VIS,
    cfg=cfg,
    dataset=dataset,
    model=model,
    k_manifest=k_manifest,
    mapping=mapping,
    rajagopal_names=rajagopal_names,
    device=DEVICE,
)
if examples:
    print(f"Vis example: {examples[-1]['sequence_id']} ({examples[-1]['smplx'].shape[0]} frames)")
else:
    print(f"Warning: could not build vis example for {VIS_SEQUENCE_ID!r}")

### Сводка по всем paired-окнам (vs OpenSim GT)

,method,n_windows,mae,rmse,dtw_mae,pearson_r_mean,r2_mean,smoothness_pred
0,MuscleMAP,617,0.01518,0.02686,0.01518,-0.0000,-36931.9626,0.00000
1,Kinesis,617,0.01649,0.02547,0.01649,0.0000,-96354.2413,0.00000


### По каждому движению (первые 25 строк)

,sequence_id,n_frames,mm_mae,kin_mae,mm_rmse,kin_rmse,mm_pearson_r,kin_pearson_r,mm_r2,kin_r2
0,Simultaneously_Nail_cutting_and_walking_clip1,256,0.01306,0.01586,0.02145,0.02240,-0.00000,0.00000,-28460.09766,-102367.10938
1,Simultaneously_Fan_yourself_and_standing_1_clip1,256,0.01308,0.01556,0.02147,0.02171,0.00000,0.00000,-30762.03516,-110740.83594
2,Simultaneously_Nail_cutting_and_walking_clip1,256,0.01308,0.01596,0.02157,0.02251,0.00000,0.00000,-30330.54102,-108051.65625
3,Simultaneously_Nail_cutting_and_walking_clip1,256,0.01311,0.01592,0.02162,0.02246,0.00000,0.00000,-30342.58594,-108070.81250
4,Simultaneously_Fan_yourself_and_standing_1_clip1,256,0.01315,0.01574,0.02167,0.02223,0.00000,0.00000,-30589.93164,-109543.84375
5,Goal_Shot_during_standing_1_clip1,128,0.01329,0.01444,0.02143,0.01915,-0.00000,0.00000,-16663.15039,-72696.75000
6,walking_while_Buying_Tickets_1_clip1,205,0.01330,0.01546,0.02168,0.02118,0.00000,0.00000,-26598.65430,-89265.85156
7,standing_while_Command_2_clip1,157,0.01336,0.01478,0.02190,0.01965,0.00000,0.00000,-28082.01172,-78634.29688
8,Exchanging_Items_while_standing_clip1,168,0.01339,0.01482,0.02180,0.02012,-0.00000,0.00000,-23067.38086,-83400.61719
9,standing_and_Rotating_the_ankle_at_the_same_time_clip1,152,0.01339,0.01610,0.02224,0.02271,-0.00000,0.00000,-25910.06250,-71532.84375


KeyboardInterrupt: 

## 4. Анимации (GT · MuscleMAP · Kinesis)

Сохраняются в `results/plots/triple_<sequence_id>.mp4` (или `.gif`, если нет ffmpeg). Параметры `VIS_FPS` и `VIS_FRAME_STEP` — в первой ячейке.

In [25]:
plots_dir = REPO_ROOT / "results" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

examples = ensure_vis_examples(
    examples,
    VIS_SEQUENCE_ID,
    max_vis=MAX_VIS,
    cfg=cfg,
    dataset=dataset,
    model=model,
    k_manifest=k_manifest,
    mapping=mapping,
    rajagopal_names=rajagopal_names,
    device=DEVICE,
)
if not examples:
    raise RuntimeError(
        f"No visualization data for {VIS_SEQUENCE_ID!r}. "
        "Run the model-load cell and check Kinesis manifest / test split."
    )

for ex in examples:
    title = f"{ex['sequence_id']}\n{ex['prompt'][:80]}"
    anim = create_triple_activation_animation(
        ex["smplx"],
        ex["gt"],
        ex["mm"],
        ex["kin_full"],
        rajagopal_names,
        fps=VIS_FPS,
        frame_step=VIS_FRAME_STEP,
        title=title,
    )
    out_path = plots_dir / f"triple_{ex['sequence_id']}.mp4"
    saved = save_animation_media(anim, out_path, fps=VIS_FPS)
    print(f"Saved {saved}")

    if saved.suffix.lower() == ".mp4":
        display(Video(str(saved), embed=True, html_attributes="controls autoplay loop"))
    else:
        display(Image(filename=str(saved)))
    display(Markdown("---"))

INFO Animation.save using <class 'matplotlib.animation.FFMpegWriter'>
INFO MovieWriter._run: running command: ffmpeg -f rawvideo -vcodec rawvideo -s 1620x576 -pix_fmt rgba -framerate 12 -loglevel error -i pipe: -vcodec h264 -pix_fmt yuv420p -b 1800k -y '/Users/maksimpecin/Library/CloudStorage/OneDrive-Личная/Thesis/code/musclemap-bench/results/plots/triple_Arm_Swing_during_walking_clip1.mp4'


Saved /Users/maksimpecin/Library/CloudStorage/OneDrive-Личная/Thesis/code/musclemap-bench/results/plots/triple_Arm_Swing_during_walking_clip1.mp4


---

## 5. Экспорт таблицы

In [ ]:
out_csv = REPO_ROOT / "results" / "kinesis_triple_comparison.csv"
metrics_df.to_csv(out_csv, index=False)
summary_csv = REPO_ROOT / "results" / "kinesis_triple_summary.csv"
summary_df.to_csv(summary_csv, index=False)
display(Markdown(f"Saved:\n- `{out_csv}`\n- `{summary_csv}`"))

Saved:
- `/Users/maksimpecin/Library/CloudStorage/OneDrive-Личная/Thesis/code/musclemap-bench/results/kinesis_triple_comparison.csv`
- `/Users/maksimpecin/Library/CloudStorage/OneDrive-Личная/Thesis/code/musclemap-bench/results/kinesis_triple_summary.csv`